# EDA — User Logs: Comportamiento de Escucha vs Churn
Agrega `user_logs.csv` (~392M filas) + `user_logs_v2.csv` por chunks, construye features  
por usuario y compara comportamiento de escucha entre churners y renewals.  

> **Nota:** la primera ejecución tarda ~10 min por el volumen del archivo (28 GB).

In [3]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.eda.user_logs_04 import (
    load_log_features,
    build_log_features,
    analyze_logs,
    plot_listening_vs_churn,
    plot_listening_trend,
    plot_churn_by_recency,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Agregación por chunks (~10 min)

In [4]:
raw_agg = load_log_features()

Agregando user_logs.csv...
  user_logs.csv: chunk 25 (50M filas)
  user_logs.csv: chunk 50 (100M filas)
  user_logs.csv: chunk 75 (150M filas)
  user_logs.csv: chunk 100 (200M filas)
  user_logs.csv: chunk 125 (250M filas)
  user_logs.csv: chunk 150 (300M filas)
  user_logs.csv: chunk 175 (350M filas)
  → 5,234,111 usuarios
Agregando user_logs_v2.csv...


ValueError: No objects to concatenate

In [ ]:
print(f'Shape agregado: {raw_agg.shape}')
raw_agg.head()

## 2. Derivar features por usuario

In [ ]:
df_features = build_log_features(raw_agg)
print(f'Features shape: {df_features.shape}')
df_features[[
    'msno', 'n_days', 'avg_daily_secs', 'avg_daily_completed',
    'completion_ratio', 'days_since_last', 'listening_trend'
]].describe()

## 3. Cruzar con etiquetas de churn

In [ ]:
DATA_RAW = ROOT / 'data' / 'raw'
labels = pd.read_csv(DATA_RAW / 'train.csv')[['msno', 'is_churn']]
print(f'Labels: {len(labels):,} usuarios | churn rate: {labels["is_churn"].mean():.2%}')

In [ ]:
results = analyze_logs(df_features, labels)

In [ ]:
# Tabla resumen con lift
summary = results['summary'].copy()
summary['lift'] = (summary['Churn (1)'] / summary['Renewal (0)']).round(2)
summary

## 4. Distribuciones por grupo: Churn vs Renewal

In [ ]:
plot_listening_vs_churn(results['merged'])

## 5. Tendencia de escucha: últimos 30 días vs histórico

In [ ]:
plot_listening_trend(results['merged'])

## 6. Churn rate por recencia del último log

In [ ]:
plot_churn_by_recency(results)

In [ ]:
results['recency_churn']

## 7. Usuarios sin logs vs churn

In [ ]:
# ¿Qué pasa con usuarios de train que NO tienen logs?
merged = results['merged']
all_train = labels.copy()
has_logs = all_train['msno'].isin(df_features['msno'])

no_logs_churn = all_train[~has_logs]['is_churn'].mean()
with_logs_churn = merged['is_churn'].mean()

fig, ax = plt.subplots(figsize=(7, 4))
groups = ['Con logs', 'Sin logs']
rates = [with_logs_churn, no_logs_churn]
colors = ['#4CAF50' if r < all_train['is_churn'].mean() else '#F44336' for r in rates]
ax.bar(groups, rates, color=colors, edgecolor='white')
ax.axhline(all_train['is_churn'].mean(), color='black', linestyle='--',
           label=f'Global {all_train["is_churn"].mean():.1%}')
for i, (g, r) in enumerate(zip(groups, rates)):
    n = len(merged) if i == 0 else (~has_logs).sum()
    ax.text(i, r + 0.002, f'{r:.1%}\n(n={n:,})', ha='center', fontsize=10)
ax.set_title('Churn rate: usuarios con y sin historial de logs')
ax.set_ylabel('Churn rate')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Resumen de hallazgos

| Feature | Renewal | Churn | Señal |
|---|---|---|---|
| `n_days` | más días activos | menos días activos | ↑ más días → menos churn |
| `avg_daily_secs` | mayor escucha/día | menor escucha/día | ↑ escucha → menos churn |
| `completion_ratio` | similar | similar | señal débil |
| `days_since_last` | próximo a 0 (activos) | alto (inactivos) | **señal fuerte** |
| `listening_trend` | estable/positivo | negativo (caída reciente) | **señal fuerte** |

**`days_since_last` es la señal más importante**: usuarios que no escucharon nada en los últimos meses tienen churn rate muy superior al promedio.